## Machine Learning ZoomCamp 2026 - Homework 3

### Setup

In [1]:
# Import required packages
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression


In [2]:
# Dataset
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
df = pd.read_csv(data)
df.head()


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


### Data preparation

* In this dataset our desired target for the classification task will be the `converted` variable - has the client signed up to the platform or not.
* Check if the missing values are presented in the features. If there are missing values:
  * For categorical features, replace them with 'NA'
  * For numerical features, replace with with 0.0


In [3]:
df.dtypes

,0
lead_source,object
industry,object
employment_status,object
location,object
annual_income,float64
number_of_courses_viewed,int64
interaction_count,int64
lead_score,float64
converted,int64


In [4]:
df.isnull().sum()

,0
lead_source,148
industry,240
employment_status,194
location,208
annual_income,369
number_of_courses_viewed,0
interaction_count,0
lead_score,35
converted,0


In [5]:
# Fill null numeric columns
df['lead_score'] = df['lead_score'].fillna(0.0)
df['annual_income'] = df['annual_income'].fillna(0.0)

# Fill null categorical columns
df['lead_source'] = df['lead_source'].fillna('NA')
df['industry'] = df['industry'].fillna('NA')
df['employment_status'] = df['employment_status'].fillna('NA')
df['location'] = df['location'].fillna('NA')


In [6]:
df.isnull().sum()

,0
lead_source,0
industry,0
employment_status,0
location,0
annual_income,0
number_of_courses_viewed,0
interaction_count,0
lead_score,0
converted,0


### Question 1

What is the most frequent observation (mode) for the column industry?

* NA
* technology
* healthcare
* retail


In [7]:
df['industry'].mode()

,industry
0,technology


### Question 2

Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

* interaction_count and lead_score
* number_of_courses_viewed and lead_score
* number_of_courses_viewed and interaction_count
* annual_income and interaction_count

Only consider the pairs above when answering this question.

**Split the data**

Split the data with these exact calls:

```
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)
```

Make sure that the target value converted is not in your dataframe.


In [36]:
# Train/validation/test set split
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)


In [37]:
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

In [38]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)


In [39]:
# Make index start at 1
df_train.index += 1
df_val.index += 1
df_test.index += 1


In [12]:
df.dtypes

,0
lead_source,object
industry,object
employment_status,object
location,object
annual_income,float64
number_of_courses_viewed,int64
interaction_count,int64
lead_score,float64
converted,int64


In [13]:
# Classify numerical and categorical values
numerical = [
    'annual_income',
    'number_of_courses_viewed',
    'interaction_count',
    'lead_score'
    ]

categorical = [
    'lead_source',
    'industry',
    'employment_status',
    'location'
    ]


In [14]:
df_full_train[numerical].corrwith(df_full_train.converted).sort_values(ascending = False)

,0
lead_score,0.482078
interaction_count,0.450273
number_of_courses_viewed,0.393200
annual_income,0.188437


### Question 3

* Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
* Round the scores to 2 decimals using round(score, 2).

Which of these variables has the biggest mutual information score?

* industry
* location
* lead_source
* employment_status


In [15]:
def mutual_info_converted_score(series):
  return mutual_info_score(series, df_train.converted)

mi = df_train[categorical].apply(mutual_info_converted_score)
mi.sort_values(ascending = False)


,0
lead_source,0.034346
employment_status,0.018966
industry,0.001995
location,0.001043


### Question 4

* Now let's train a logistic regression.
* Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
* Fit the model on the training dataset.
* To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:

    ```model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)```

* Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?

* 0.55
* 0.65
* 0.75
* 0.85


In [40]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']


In [17]:
# One-hot encoding
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)


In [18]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)


LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [19]:
y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
round((y_val == converted_decision).mean(), 2)

np.float64(0.64)

### Question 5

* Let's find the least useful feature using the feature elimination technique.
* Train a model using the same features and parameters as in Q4 (without rounding).
* Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
* For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

Which of following feature has the smallest difference?

* 'lead_source'
* 'number_of_courses_viewed'
* 'interaction_count'

> Note: The difference doesn't have to be positive.




In [20]:
y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
model_full_acc = (y_val == converted_decision).mean()
model_full_acc

np.float64(0.645)

In [44]:
cols = categorical + numerical

for i in cols:
    # Create a copy of the columns list and remove the current feature
    subset_cols = [c for c in cols if c != i]

    dv = DictVectorizer(sparse=False)

    train_dict = df_train[subset_cols].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)

    val_dict = df_val[subset_cols].to_dict(orient='records')
    X_val = dv.transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    converted_decision = (y_pred >= 0.5)
    model_acc = (y_val == converted_decision).mean()

    difference = model_full_acc - model_acc
    print(f"Feature excluded: {i}, Accuracy: {model_acc}, Difference: {difference}")

Feature excluded: lead_source, Accuracy: 0.642, Difference: 0.0030000000000000027
Feature excluded: industry, Accuracy: 0.645, Difference: 0.0
Feature excluded: employment_status, Accuracy: 0.645, Difference: 0.0
Feature excluded: location, Accuracy: 0.645, Difference: 0.0
Feature excluded: annual_income, Accuracy: 0.724, Difference: -0.07899999999999996
Feature excluded: number_of_courses_viewed, Accuracy: 0.643, Difference: 0.0020000000000000018
Feature excluded: interaction_count, Accuracy: 0.601, Difference: 0.04400000000000004
Feature excluded: lead_score, Accuracy: 0.644, Difference: 0.0010000000000000009


### Question 6

* Now let's train a regularized logistic regression.
* Let's try the following values of the parameter `C`: `[0.000001, 0.00001, 0.0001, 0.001]`.
* Train models using all the features as in Q4.
* Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these `C` leads to the best accuracy on the validation set?

* 0.000001
* 0.00001
* 0.0001
* 0.001

> Note: If there are multiple options, select the smallest C.


In [49]:
# Regularization parameters
C_values = [0.000001, 0.00001, 0.0001, 0.001]

# One-hot encoding
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

for c in C_values:
    # Model training
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    # Model prediction
    y_pred = model.predict_proba(X_val)[:, 1]
    converted_decision = (y_pred >= 0.5)
    acc = (y_val == converted_decision).mean()
    print(f"C: {c}, Accuracy: {round(acc, 3)}")


C: 1e-06, Accuracy: 0.598
C: 1e-05, Accuracy: 0.598
C: 0.0001, Accuracy: 0.613
C: 0.001, Accuracy: 0.645
